# Tutorial: Extracting Unstructured Text Using Large Language Models

The following code is identical as displayed in Spavound, S., and Schaer, O., and Markou, P.,, Tutorial: Extracting Unstructured Text Using Large Language Models (April 10, 2026). Available at SSRN: https://ssrn.com/abstract=6556303

Note: A full PDF extraction example with an implementation of a chunking algorithm will be available soon.

---

## Tutorial and Large Language Model Building Blocks

In [ ]:
# Listing 1
from openai import OpenAI

#client = OpenAI(api_key="your_api_key_here")
client = OpenAI()

response = client.responses.create(
    model="gpt-5.4-2026-03-05",
    input="What are the FDA's key purposes?"
)

print(response.output_text)

In [ ]:
# Listing 2

from pydantic import BaseModel, Field
from typing import List

# Define schema for each row
class FDAPurposeItem(BaseModel):
    purpose: str = Field(
        description="A short phrase summarizing one core purpose of the FDA."
    )
    explanation: str = Field(
        description="A single concise sentence explaining the purpose."
    )

# Define overall response schema
class FDAPurposes(BaseModel):
    items: List[FDAPurposeItem] = Field(
        description="A list of FDA purposes, each with a short label and explanation."
    )

# Call API with structured parsing
response = client.responses.parse(
    model="gpt-5.4-2026-03-05",
    input="What are the FDA's 3 key purposes?",
    text_format=FDAPurposes
)

In [ ]:
# Optional print statement for Listing 3 and Listing 4
print(response.output_text)

In [ ]:
# Listing 5

response = client.responses.parse(
    model="gpt-5.4-2026-03-05",
    input="What are the FDA's 3 key purposes?",
    temperature=0.0,
    text_format=FDAPurposes
)

In [ ]:
# Optional print statement for Listing 6 and Listing 7
print(response.output_text)

In [ ]:
# Listing 8

import base64
from pathlib import Path

# Path to your image
image_path = Path("3954t1-FDA_Page_37.png")

with open(image_path, "rb") as f:
     base64_image = base64.b64encode(f.read()).decode()

# Define schema for each row
class GraphItem(BaseModel):
    label: str = Field(
        description="The exact text of the headings as it appears in the graph."
    )
    type: str = Field(
        description="Identifier of the structural element this text represents"
    )

# Define overall response schema
class GraphLabels(BaseModel):
    items: List[GraphItem] = Field(
        description="A list of descriptive headings extracted from the graph."
    )

# Call API with structured parsing
response = client.responses.parse(
    model="gpt-5.4-2026-03-05",
  input=[{
        "role": "user",
        "content": [
            {"type": "input_text", 
             "text": "Extract and identify the descriptive headings from this slide"},
            {"type": "input_image",
             "image_url": f"data:image/jpeg;base64,{base64_image}"}
        ]
    }],
    text_format=GraphLabels
)

In [ ]:
# Optional print statement for Listing 9

print(response.output_text)

---

## Building a Pipeline for Obtaining Structured Data from Unstructured Text

In [ ]:
# Listing 10

from pdf2image import convert_from_path
from pathlib import Path
from openai import OpenAI
import base64

pdf = Path("example.pdf") # input PDF
out = Path("example") # output folder
out.mkdir(parents=True, exist_ok=True) # create folder if it does not exist

images = convert_from_path(pdf, output_folder=out, fmt="png", output_file=pdf.stem)

# We load and encode one single page into base64
image_path = Path("example/example0001-061.png")
with open(image_path, "rb") as f:
     base64_image = base64.b64encode(f.read()).decode()

# Initialize OpenAI client
#client = OpenAI(api_key="your_api_key_here")
client = OpenAI()

# Image extraction prompt
prompt = """You are a professional OCR engine. Transcribe the text EXACTLY as written.
Do not correct spelling or grammar errors.
If text is blurry use context to infer the word.
Ignore headers footers as well as line and page numbers.
"""

# Image extraction call
response = client.responses.create(
    model="gpt-4.1-mini-2025-04-14",
    temperature = 0.0,
    input=[
        {
            "role": "user",
            "content": [
                { "type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{base64_image}",
                },
            ],
        }
    ],
)

raw_text = response.output_text

In [ ]:
# Optional print statement for Listing 12

print(response.output_text)

In [ ]:
# Listing 13
from pydantic import ConfigDict, BaseModel, Field
from typing import List

# Pydantic class for speaker-statement pairs
class TranscriptEntry(BaseModel):
    model_config = ConfigDict(extra="forbid")

    speaker: str = Field(
        description="The speaker's name in UPPERCASE. Use 'UNKNOWN' if the text starts without a name. Do not extract names from headers or rosters."
    )
    statement: str = Field(
        description="The spoken text verbatim, including all bracketed [...] or parenthetical (...) notes like [Laughter.], (Applause.). Remove all line breaks (\\n) so it is a single line."
    )

# Pydantic schema class
class TranscriptResponse(BaseModel):
    model_config = ConfigDict(extra="forbid")

    entries: List[TranscriptEntry] = Field(
        description="A list of transcript entries."
    )

In [ ]:
# Listing 14

# Text structuring prompt
system_prompt = """You are an expert transcript cleaner. You will receive a raw text chunk. Your goal is to extract the dialogue into a structured format.

**PRIME DIRECTIVE - WORD FIDELITY VS. FORMAT FLEXIBILITY:**
1. **DO NOT** change words, fix grammar,  remove non-verbal markers (e.g., [Laughter.], (Applause.)) or autocomplete sentences.
2. **DO** fix whitespace. You must remove line breaks within a sentence and normalize multiple spaces into a single space.

Follow these processing rules:

1. **Scope & Exclusions:**
   - **Ignore Metadata:** Do not extract meeting rosters, attendee lists, or headers. Start extracting only when the actual dialogue begins.
   - **Ignore Artifacts:** Remove page numbers, file paths, and margin line numbers.

2. **Speaker Identification:**
   - **Standard Speech:** Extract the speaker's name exactly as written and convert to UPPERCASE.
   - **Orphaned Speech:** If the chunk starts with sentences/dialogue but has no speaker name attached, label the speaker as: UNKNOWN."""

# text structuring call
response = client.responses.parse(
    model="gpt-4.1-mini-2025-04-14",
    temperature=0.0,
    input=[
        {"role": "system", "content": system_prompt},  # Prompt instructions
        {"role": "user", "content": raw_text}          # Section of raw text
    ],
    text_format=TranscriptResponse
)

In [ ]:
# Print listing 15

print(response.output_text)